# Lab 2 — Getting data in
### EP0418 · Topic 2 · 2 practical hours (Week 2)

**You need:** `sg_rainfall_daily.csv`, `rain_now.json` and `ep0418.py`, which the
notebook fetches, and the Lab 2 sheet.

---
Singapore has about ninety rain gauges, and the government publishes what each one
measured every five minutes. Today that data arrives in the three ways the Topic 2B
notes describe, and you meet all three:

- **a CSV**: a saved table, sixty days of rain, one row per day;
- **JSON**: what the live service actually sends, boxes inside boxes, one
  five-minute snapshot of every gauge;
- **a live API**: the service itself, which will surprise you.

Then you do what Topic 2A is about: **plan first**, then prompt, generate, verify,
refine, to turn the JSON into a table you can ask a question of. Each task below
says which part of Topic 2 it practises. This lab is the same weight as Lab 1.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

import json
fetch("sg_rainfall_daily.csv", "rain_now.json")

## Task 1 — A saved table (CSV), and the five questions (20 minutes)

*Topic 2B: the five questions to ask of any data you have never seen.*

A CSV is a table saved as text. Load it and ask the five questions from the notes,
in order, before doing anything clever.

In [ ]:
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

print("1. Rows and columns:", rain.shape, list(rain.columns))
print("2. Period:", rain.date.min().date(), "to", rain.date.max().date(),
      "| largest gap between rows:", rain.date.diff().max().days, "day(s)")
print("3. Smallest and largest, each column:")
print(rain.describe().loc[["min", "max"]].round(2))
print("4. Blanks:", int(rain.isna().sum().sum()), "| days with zero rain everywhere:", int((rain.rain_mm_mean == 0).sum()))
print("5. Anything impossible? negative rain:", int((rain[["rain_mm_mean", "rain_mm_max"]] < 0).sum().sum()))
show.series(rain, col="rain_mm_mean", title="Average rain across all gauges, mm per day")

**Expected output** *(computed from the data file when this notebook was built)*: Row 1: 60 rows, four columns. Row 2: every day present, largest gap 1 day.
Row 4: no blanks; 3 completely dry days. Row 5: no negative rain.

**What you should see.** Sixty days, no gaps, no blanks. Now look at the "max"
line: the wettest single reading is far above the daily average. Rain in
Singapore falls hard in one place for twenty minutes while the rest of the island
is dry. And `stations_reporting` is not the same every day.

**Write it down:** the strangest number on screen, and what you think caused it.

**Strangest number:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Two good answers. The largest `rain_mm_max` is enormous next to the average,
because it is one gauge under one storm cloud while the average is spread over
ninety gauges, most of them dry. And `stations_reporting` moves between two values:
on some days one gauge sent nothing. That is not an error in the file; it is a
gauge that was offline, and it is the first hint that "missing" and "zero" are
different things. Lab 4 is about that difference.
</details>

## Task 2 — Boxes inside boxes (JSON), and the live feed (20 minutes)

*Topic 2B: JSON is not a table, and a live API will surprise you.*

`rain_now.json` is one five-minute snapshot, saved exactly as the live service sent
it. Run the cell and read the shape before you read the numbers.

In [ ]:
raw = json.load(open("rain_now.json"))["data"]

print("top-level boxes:", list(raw.keys()))
print("how many stations:", len(raw["stations"]), "| first one:", raw["stations"][0])
reading = raw["readings"][0]
print("the reading is from:", reading["timestamp"])
print("how many values in it:", len(reading["data"]), "| first three:", reading["data"][:3])

**What you should see.** Two lists that belong together but are stored apart: the
stations (id, name, location) and the readings (station id, value). Count them:
there are more stations than values. Two gauges sent nothing in this snapshot. To
answer "which gauge was wettest" you have to join the lists by station id without
losing those two, and without pretending they measured zero. That is the work
Task 3 asks Gemini to do.

**Now the live service.** The cell below asks for the snapshot being taken right
now. Run it once. It may work; it may answer 403 (refused, no reason given) or 429
(too many requests, forty of you at once). Either way, read the message: the notes
list exactly these three surprises, and you have just met one. Nothing later
depends on this cell.

In [ ]:
try:
    req = urllib.request.Request("https://api-open.data.gov.sg/v2/real-time/api/rainfall",
                                 headers={"User-Agent": "EP0418 lab"})
    live = json.load(urllib.request.urlopen(req, timeout=20))["data"]
    vals = [x["value"] for x in live["readings"][0]["data"]]
    print("live:", live["readings"][0]["timestamp"], "|", len(vals), "gauges |",
          sum(v > 0 for v in vals), "reporting rain right now | wettest", max(vals), "mm")
except Exception as e:
    print("The service surprised you:", e)
    print("That is normal (see the notes: pagination, rate limits, silent rejection). "
          "Build on the saved file; demonstrate on the feed.")

## Task 3 — Plan first, then the loop (45 minutes)

*Topic 2A: plan, prompt, generate, verify, refine. "It ran" is the weakest evidence.*

**The goal:** one table with one row per gauge: `name` and `rain_mm`. Then the
wettest ten.

**Plan first.** Before any code, ask Gemini to think with you in plain words:

> I am new to this. Answer in three plain sentences, no headings, no bullet
> points, no code. I have JSON from a rain service with two lists: "stations"
> (each has id and name) and "readings" (each has a timestamp and a list of
> stationId and value). How would you turn one reading into a table with one row
> per station, showing its name and its value? What could go wrong?

If an essay with headings comes back, reply **"simpler, three sentences"**. That
is the refine step; assistants over-answer, and telling them the shape you want is
normal. The plan should mention matching the two lists by station id. If it
does not, ask: *how do you know which name goes with which value?*

**Prompt** for the code, rules first. In Colab, `raw` already exists:

> Do not drop any rows. Show the row count before and after. Explain in plain
> words what the code does. `raw` is a Python dict with `raw["stations"]`, a list
> of dicts with keys `id` and `name`, and `raw["readings"][0]["data"]`, a list of
> dicts with keys `stationId` and `value`. Write pandas code that builds a
> DataFrame `table` with one row per station and two columns, `name` and
> `rain_mm`, matching each value to its station by id. A station with no value
> must keep its row, with `rain_mm` left blank (NaN), not 0.

**Generate** is Gemini's job. **Verify** is yours: paste the code into the cell in
place of the `TODO` line, run it, read the checks. **Refine**: if a check fails,
tell Gemini which one and what it said.

In [ ]:
table = ...     # TODO(you): paste Gemini's code above this line so that it creates `table`

# ---- verification contract — do not edit ----
if filled(table):
    check.columns_present(table, ["name", "rain_mm"])
    check.rows_at_least(table, len(raw["stations"]))      # one row per station: a join that lost the silent gauges fails here
    check.values_in_range(table, "rain_mm", 0, 200)
    check.report()
    print("stations:", len(raw["stations"]), "| values in the reading:", len(raw["readings"][0]["data"]),
          "| rows in your table:", len(table), "| blank rain_mm:", int(table.rain_mm.isna().sum()),
          "| zero rain_mm:", int((table.rain_mm == 0).sum()))

**Read that last line.** Three outcomes are possible, and only one is right:
77 rows with 2 blanks (right: the silent gauges are there, marked unknown);
75 rows (wrong: the join quietly dropped them, and `rows_at_least` says so);
77 rows with 0 blanks (wrong, and no check catches it: "no reading" has become
"no rain"). If you got the third, tell Gemini: *the two stations with no value
must be NaN, not 0*. That is the refine step, and it is the whole of Lab 4 in
one line.

**Check two rows by hand.** The notes say verify means: did it run, is the shape
right, is the range right, and can two rows be checked by hand. The checks did the
first three. Now the fourth: pick any station in the raw JSON printout from Task 2,
find its id in `raw["readings"][0]["data"]`, and confirm the table shows the same
number against that name. The cell helps.

In [ ]:
if filled(table):
    sid = raw["stations"][0]["id"]
    by_hand = [x["value"] for x in raw["readings"][0]["data"] if x["stationId"] == sid]
    print("station", sid, raw["stations"][0]["name"], "| value in the raw JSON:", by_hand)
    print("the same station in your table:")
    print(table[table.name == raw["stations"][0]["name"]])
    print()
    print("Wettest ten gauges in this snapshot:")
    top = table.nlargest(10, "rain_mm")
    print(top.to_string(index=False))
    top.iloc[::-1].plot.barh(x="name", y="rain_mm", legend=False, figsize=(7, 4),
                             title="Rain in the last five minutes, mm"); plt.tight_layout(); plt.show()

**Write it down:** did the hand-checked row match? How many gauges had any rain at
all, and where was the wettest one?

**Hand check and wettest gauge:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

The hand-checked value and the table value must be identical; if they are not, the
join matched the wrong id to the wrong name, which is the classic silent fault in
this task, and no check above would have caught it. In this snapshot (a morning in
late May) about a dozen of the 77 gauges show rain, the wettest just under 6 mm in
five minutes, clustered in one part of the island: a storm is small and the island
is not. Two gauges have no value at all; they must appear as blanks, not zeros.
</details>

---
## Read before you run

An assistant was asked which rain gauge was wettest in the snapshot. It wrote the code below, which ran and printed a station name.

**Do not run it yet.** Read it, answer the two questions below,
and only then remove the `#` and run it to see whether you were right.

Which row does `.iloc[0]` pick after the sort: the smallest value or the largest? What will it print on a snapshot where most gauges read zero?

Write down what the code prints, and what it should print (compare with the wettest-ten list from Task 3).

<details><summary>Hint 1: where to look</summary>

Which way does `sort_values` sort unless you tell it otherwise: smallest first, or largest first?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Smallest first. Row 0 after the sort is the driest gauge, not the wettest. The code runs, prints a real station name, and is wrong. Two rows checked by hand would show it.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# --- AI-generated. Read it first. ---
# wettest = table.sort_values("rain_mm").iloc[0]
# print("wettest gauge:", wettest["name"], wettest["rain_mm"], "mm")

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the three ways data arrives (CSV, JSON, live API) and what goes wrong with each; the five questions to ask of any new dataset, in order; why JSON is not a table and what joining two lists by id means; the four steps of the loop (prompt, generate, verify, refine) and the step most people skip; why 'it ran without an error' is weak evidence, and what checking two rows by hand adds. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- The strangest number in the CSV, and what caused it.
- Which API surprise you met, if any.
- One sentence: what "plan first" changed about the code Gemini wrote.

**Mistake caught:** *(what I expected · what was there · how I found it)*